# Autoencoders (autocodificadores)

Detector de fraude sin etiquetas: el error de reconstrucción delata las compras raras.

[Abrir en el explorador](https://stivenson.github.io/#/articles/algoritmos-ml-explorador?alg=autoencoders&tab=realWorld)

Ejecuta la celda de abajo (no hace falta GPU).

In [ ]:
# Detector de fraude sin etiquetas: un autoencoder que aprende cómo es una compra normal
import numpy as np

rng = np.random.default_rng(7)
# 6 medidas por compra. En las compras normales salen de 2 factores ocultos
# (cuánto se gasta y cuán lejos de casa se compra), más un poco de ruido.
mezcla = rng.normal(size=(2, 6))
normales = rng.normal(size=(1000, 2)) @ mezcla + 0.2 * rng.normal(size=(1000, 6))
fraudes = rng.normal(size=(10, 6)) * 1.5  # no siguen el patrón de las normales
# Estandarizar con la media y la desviación de las normales (también a los fraudes),
# para que ninguna medida tape a las demás. Así los datos quedan con media 0: no hace falta sesgo.
media, desv = normales.mean(axis=0), normales.std(axis=0)
normales = (normales - media) / desv
fraudes = (fraudes - media) / desv
print(f"{len(normales)} compras normales para entrenar y {len(fraudes)} fraudes que el modelo nunca ve")


def entrenar(X, k, pasos=3000, tasa=0.05):
    """Autoencoder lineal: codifica 6 números en k (cuello de botella) y los decodifica de vuelta."""
    r = np.random.default_rng(0)
    W_cod = 0.1 * r.normal(size=(6, k))
    W_dec = 0.1 * r.normal(size=(k, 6))
    for _ in range(pasos):  # descenso de gradiente sobre el error cuadrático de reconstrucción
        Z = X @ W_cod
        dif = Z @ W_dec - X
        grad_dec = Z.T @ dif / len(X)
        grad_cod = X.T @ (dif @ W_dec.T) / len(X)
        W_dec -= tasa * grad_dec
        W_cod -= tasa * grad_cod
    return W_cod, W_dec


def error(X, W_cod, W_dec):
    return np.mean((X @ W_cod @ W_dec - X) ** 2, axis=1)  # error de reconstrucción de cada compra


print("\ncuello k | error normales | error fraudes | fraudes detectados")
for k in (1, 2, 4):
    W_cod, W_dec = entrenar(normales, k)
    e_norm, e_fraude = error(normales, W_cod, W_dec), error(fraudes, W_cod, W_dec)
    umbral = np.quantile(e_norm, 0.99)  # alarma si el error supera al del 99 % de las normales
    print(f"{k:8d} | {e_norm.mean():14.2f} | {e_fraude.mean():13.2f} | {np.sum(e_fraude > umbral):9d} de {len(fraudes)}")

**Opcional: con PyTorch.** La celda siguiente hace lo mismo con PyTorch, la biblioteca que se usa en la práctica. Colab ya lo trae instalado; si no está, la celda solo avisa y no falla. Ejecuta antes la celda del ejercicio.

In [ ]:
# Opcional: un autoencoder no lineal en PyTorch (Colab ya lo trae instalado; en el navegador no corre).
# Usa `normales` y `fraudes` de la celda anterior. Sus cifras no tienen por qué coincidir con las del lineal.
try:
    import torch
    from torch import nn
except (ImportError, OSError):  # OSError: PyTorch instalado pero roto (falta una biblioteca)
    print("PyTorch no está instalado: esta celda es opcional.")
else:
    torch.manual_seed(0)
    N_pt = torch.tensor(normales, dtype=torch.float32)
    F_pt = torch.tensor(fraudes, dtype=torch.float32)
    modelo = nn.Sequential(  # 6 → 4 → 2 (cuello de botella) → 4 → 6
        nn.Linear(6, 4), nn.Tanh(), nn.Linear(4, 2),
        nn.Linear(2, 4), nn.Tanh(), nn.Linear(4, 6),
    )
    opt = torch.optim.Adam(modelo.parameters(), lr=0.01)
    for _ in range(2000):
        opt.zero_grad()
        perdida = ((modelo(N_pt) - N_pt) ** 2).mean()
        perdida.backward()
        opt.step()
    with torch.no_grad():
        e_norm = ((modelo(N_pt) - N_pt) ** 2).mean(dim=1)
        e_fraude = ((modelo(F_pt) - F_pt) ** 2).mean(dim=1)
    umbral = torch.quantile(e_norm, 0.99)
    print(f"PyTorch, cuello de 2: detecta {(e_fraude > umbral).sum().item()} de {len(F_pt)} fraudes")